# Setup (No Training Here)


## Imports

In [ ]:
using ComponentArrays
using Random
using LinearAlgebra
using SparseArrays
using LinearSolve
using OrdinaryDiffEq
using OrdinaryDiffEqSDIRK
using OrdinaryDiffEqLowOrderRK
using SciMLSensitivity
using ADTypes
using Zygote
using Enzyme
using Optimization
using OptimizationOptimisers
using OptimizationOptimJL
using LineSearches
using Lux
using Functors
using Plots

# Enzyme.API.runtimeActivity!(true)
Enzyme.set_runtime_activity(Enzyme.Reverse)


# ### ADJUSTED: Load local training helpers from Research_Code/src.
include(joinpath(@__DIR__, "..", "..", "src", "Misc.", "misc_helpers.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Local", "integration_AC.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Local", "FOM_opt_AC.jl"))

## Forward Solve

In [ ]:
# parameters
N = 256
L = 1.0
Δx = L/(N+1)
x = L*Δx*collect(1:N)

ε2 = 1e-2
k = 1.0
p0 = ComponentVector(ε2=ε2, k=k, Δx=Δx)
u0 = tanh.((x .- L/2) / sqrt(2ε2))

tspan = (0.0, 2.0)
@show Δt = 0.5 * Δx^2/(2ε2)
### ADJUSTED: Cap saved reference times to match the HPC common reference builder.
save_count = min(max(2, floor(Int, (tspan[2] - tspan[1]) / Δt) + 1), 500)
t = LinRange(tspan[1], tspan[2], save_count)

#f = ODEFunction(rhs_ac!)
# jac_prototype builts a prototype matrix for the jacobian of the system
# in this case, tridiagonal; the free energy term is pointwise, and the laplacian is tridiagonal
# but do we even need the jacobian if we're using an Euler solver?? idts. mainly for implicit (eg TRBDF2) solver
f = ODEFunction(rhs_ac!; jac_prototype=Tridiagonal(zeros(N-1),zeros(N),zeros(N-1)))

prob = ODEProblem(f, u0, tspan, p0)

alg = Euler()
# set up an adjoint solver. autojacvec will help the solver differentiate through the RHS
# note that this isn't just reverse mode AD. Gauss has to build an adjoint of the true rhs so that it can integrate however it needs to
# however, in this case, the RHS is discrete (discrete laplacian + FE eval)
sensalg = GaussAdjoint(autojacvec=EnzymeVJP())

# solve the forwards problem
@time u_ref = solve(prob, alg; dt=Δt, saveat=t)
@time u_ref = solve(prob, alg; dt=Δt, saveat=t);


In [ ]:
@gif for i in 1:length(u_ref.t)
    plot(x, u_ref.u[i], lw=2, ylim=(-1,1), label="")
end every 64

## Neural PDE Training

In [ ]:
f_true(u) = u^3 - u
plot(u -> f_true(u), xlim=(-1,1), ylim=(-0.5,0.5), lw=2, label="True", title="Learned vs true FE derivatives")

In [ ]:
(; prob, p₀, t_obs, nn, state, x, u₀, run_params) =
    prepare_for_optimization(
        N=256,
        L=1.0,
        ε2=1e-2,
        tspan=(0.0, 2.0),
        N_obs=10,
        h=8,
        seed=1,
    )

In [ ]:
optprob = set_up_optimization(
    u_ref,
    prob,
    t_obs,
    p₀;
    run_params,
)


In [ ]:
output1 = run_full_optimization(
    optprob;
    η=5e-3,
    β=(0.9, 0.99),
    N_iter=500,
    warmup=true,
    save_frequency=10,
)

optprob_2 = remake(optprob; u0=output1.result.u)


output2 = run_full_optimization(
    optprob_2;
    η=1e-3,
    β=(0.9, 0.99),
    N_iter=800,
    warmup=false,
    save_frequency=10,
)

optprob_3 = remake(optprob_2; u0=output2.result.u)


output3 = run_full_optimization(
    optprob_3;
    η=1e-4,
    β=(0.9, 0.99),
    N_iter=1100,
    warmup=false,
    save_frequency=10,
)

output = output3

In [ ]:
# run_directory = save_optimization_data(output, run_name)

## Visualize 

In [ ]:
result = output.result
evaluation_history = output.evaluation_history
nn = prob.f.f.nn
state = prob.f.f.state

θ_opt = result.u
_, re = Optimisers.destructure(p₀.θ)
ps_opt = re(θ_opt)

us = range(-1, 1; length=200)
Fvals = [Fnn(u, nn, ps_opt, state) for u in us]

plot(
    u -> -k * (u^3 - u);
    xlim=(-1, 1),
    ylim=(-0.7, 0.7),
    lw=2,
    label="True",
)

plot!(us, Fvals;
    lw=2,
    label="Neural network",
    ls=:dash,
)

In [ ]:
iterations = getproperty.(evaluation_history, :iteration)
losses = getproperty.(evaluation_history, :loss)
evaluation_counts = getproperty.(evaluation_history, :count)

loss_plot = plot(
    iterations,
    losses;
    xlabel="Optimization iteration",
    ylabel="Loss",
    title="Loss history",
    lw=2,
    marker=:circle,
    label=false,
    yscale=:log10,
)

evaluation_plot = plot(
    iterations,
    evaluation_counts;
    xlabel="Optimization iteration",
    ylabel="Model evaluations",
    title="Function evaluations per callback",
    lw=2,
    marker=:circle,
    label=false,
)

plot(loss_plot, evaluation_plot; layout=(2, 1), size=(800, 700))

In [ ]:
iterations = getproperty.(evaluation_history, :iteration)
evaluation_counts = getproperty.(evaluation_history, :count)
cumulative_evaluations = cumsum(evaluation_counts)

plot(
    iterations,
    cumulative_evaluations;
    xlabel="Optimization iteration",
    ylabel="Cumulative Fnn evaluations",
    title="Total Fnn evaluations over optimization",
    lw=2,
    marker=:circle,
    label=false,
)